# 🧠 Experiments: Mem0 Customer Memory and a LangChain Tool Agent

## Learning Objectives
In this notebook, you will learn:
1. **Mem0 memory** - store facts from a conversation and recall them per user with semantic search
2. **Per-user isolation** - see that one customer's memories never answer another customer's query
3. **A LangChain tool agent** - build a `create_agent` agent on a Groq model and watch it decide when to call a tool

## Where this fits
These are the two building blocks of the support copilot. `customer_support_agent/integrations/memory/mem0_store.py` wraps the same Mem0 calls, and `customer_support_agent/services/copilot_service.py` drafts replies with a Groq model.

## Prerequisites
- Agent basics (repository 06, agent fundamentals) - as reading only
- Dependencies installed with `uv sync` in this project's folder (Python 3.11)
- `GROQ_API_KEY` (Mem0's fact extraction and the agent) and `GOOGLE_API_KEY` (Gemini embeddings for Mem0) in this project's `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads the keys from this project's `.env`, suppresses warnings and stops early if a key is missing.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads GROQ_API_KEY and GOOGLE_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

for key in ("GROQ_API_KEY", "GOOGLE_API_KEY"):
    if not os.getenv(key):
        raise ValueError(f"Set {key} in this project's .env before running this notebook.")

print("✅ Environment ready")

---
## 🗂️ Part 1: Mem0 Memory

Mem0 turns conversations into short, searchable facts. On `add`, an LLM (here Groq `llama-3.3-70b-versatile`) extracts facts from the messages; an embedder turns each fact into a vector; a vector store keeps them, tagged with the `user_id`. On `search`, the query is embedded and matched against that user's facts only.

### Key Concepts:
- **LLM**: extracts and updates facts (Groq)
- **Embedder**: Gemini `gemini-embedding-001` at 768 dimensions, the same provider the copilot uses
- **Vector store**: a local Qdrant collection on disk under `./data/mem0-qdrant` (git-ignored)

### 1.1 ⚙️ Configure and create the memory

In [ ]:
# ============================================================================
# MEM0 CONFIG: Groq LLM + Gemini embedder + local Qdrant store
# ============================================================================
mem0_config = {
    "llm":{"provider": "groq",
        "config": {
            "model": "llama-3.3-70b-versatile",
            "api_key": os.environ["GROQ_API_KEY"],
            "temperature": 0,
        },},
    "embedder":{"provider": "gemini",
        "config": {
            "model": "gemini-embedding-001",
            "api_key": os.environ["GOOGLE_API_KEY"],
            "embedding_dims": 768,
        },},
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "path": "./data/mem0-qdrant",
            "embedding_model_dims": 768,  # must match the embedder's embedding_dims
        },}
}

print("✅ Mem0 config ready")

In [ ]:
# ============================================================================
# MEMORY: Build the Mem0 instance from the config
# ============================================================================
from mem0 import Memory

m = Memory.from_config(mem0_config)

print("✅ Mem0 memory ready")

### 1.2 ✍️ Remember a customer

`add` sends Alex's conversation to the LLM, which extracts facts such as the name and interests. The return value lists each memory event (`ADD`, `UPDATE`, ...).

In [ ]:
# ============================================================================
# ADD: Store facts from Alex's conversation
# ============================================================================
messages = [
    {"role": "user", "content": "Hi, I'm Alex. I love basketball and gaming."},
    {"role": "assistant", "content": "Hey Alex! I'll remember your interests."}
]
m.add(messages, user_id="alex")

In [ ]:
# ============================================================================
# SEARCH: Recall what we know about Alex
# ============================================================================
results = m.search("What do you know about me?", user_id= "alex")
print(results)

### 1.3 🔒 Memories are scoped per user

Nothing has been stored for Yash yet, so the same query with `user_id="yash"` should return no results, even though Alex's facts are in the same store.

> **Note**: the store persists on disk. On a re-run, Yash's memories from the previous run are already there; delete `notebooks/data/mem0-qdrant/` to start from an empty store.

In [ ]:
# ============================================================================
# SEARCH: Same query for a user with no memories yet
# ============================================================================
results = m.search("What do you know about me?", user_id="yash")
print(results)

In [ ]:
# ============================================================================
# ADD: Store facts from Yash's conversation
# ============================================================================
messages = [
    {"role": "user", "content": "Hi, I'm Yash. I love chess and programming."},
    {"role": "assistant", "content": "Hey Yash! I'll remember your interests."}
]
m.add(messages, user_id="yash")

In [ ]:
# ============================================================================
# SEARCH: Now Yash has his own memories, and only his
# ============================================================================
results = m.search("What do you know about me?", user_id="yash")
print(results)

Compare the two Yash searches: the first returned nothing, the second returns only Yash's chess and programming facts, never Alex's basketball and gaming.

---
## 🤖 Part 2: A LangChain Tool Agent on Groq

The copilot drafts replies with a Groq model. Here we call the model directly, then wrap it in a LangChain 1.x `create_agent` agent with one tool and compare a message that needs no tool with one that does.

### 2.1 💬 Call the Groq model

In [ ]:
# ============================================================================
# LLM: Groq llama-3.1-8b-instant (kept: this project has no aied-helpers, Python 3.11)
# ============================================================================
from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=128)

llm_groq.invoke("hi! i am yash")

### 2.2 🛠️ Define a tool and build the agent

`@tool` turns a typed function into a tool; its docstring becomes the description the model reads to decide when to call it.

In [ ]:
# ============================================================================
# TOOL: A trivial uppercase tool, just for testing
# ============================================================================
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def dummy_tool(inputText:str)-> str:
    """Convert input text to uppercase. This is just for testing.
     Arguments:
        inputText {str} -- input text to be converted to uppercase
    """
    return f"Outputs : {inputText.upper()}"


my_tools = [dummy_tool]
print(f"✅ Tool ready: {dummy_tool.name}")

In [ ]:
# ============================================================================
# AGENT: create_agent wires the model and tools into a ReAct loop
# ============================================================================
my_agent = create_agent(model=llm_groq,tools=my_tools)

my_agent

### 2.3 🔁 With and without a tool call

A greeting needs no tool, so the agent should answer directly. Asking to uppercase text should make it call `dummy_tool` and use its output. Each message in the result shows its type, so you can see whether a `tool` message appears.

In [ ]:
# ============================================================================
# RUN 1: A message that needs no tool
# ============================================================================
res = my_agent.invoke({"messages":[("user","hi! i am yash")]})

for msg in res["messages"]:
    msg.pretty_print()

In [ ]:
# ============================================================================
# RUN 2: A message that should trigger the tool
# ============================================================================
res_with_tool = my_agent.invoke({"messages":[("user","hi! i am yash and i want to convert `Ai agent` to uppercase")]})

for msg in res_with_tool["messages"]:
    msg.pretty_print()

In run 2, look for an AI message with a `dummy_tool` tool call, then a tool message `Outputs : AI AGENT`, then the final answer. Run 1 has only the human and AI messages.

---
## 📝 Summary

In this notebook, we learned:

### 1. Mem0 memory
- `Memory.from_config` combines an LLM (fact extraction), an embedder and a vector store; `add` stores facts, `search` recalls them.

### 2. Per-user isolation
- Every memory is tagged with a `user_id`, and `search` only sees that user's facts, which is how the copilot keeps customers apart.

### 3. A LangChain tool agent
- `create_agent` with `@tool` functions lets the model decide per message whether a tool call is needed.

### Next Steps
- **The project's app**: the FastAPI service (`uv run python main.py`) and the Streamlit dashboard (`uv run streamlit run app.py`) combine Mem0 memory, knowledge-base RAG and Groq drafts. See the project README.